<h1><font color='yellow'>Workbook 2</font>

<h2><font color="red">Data Ingestion Challenges I</font></h2>

<p><font color="blue"><b>Run the cell below. This will drop a .txt file named log_entries.txt into the folder into your working folder (where this notebook is).</b></font></p>

In [1]:
import random
from datetime import datetime, timedelta

def generate_system_log(lines=1000):
    """Generate fake system log data and return as a string"""
    log_levels = ['INFO', 'DEBUG', 'WARNING', 'ERROR']
    services = ['web_server', 'database', 'auth_service', 'file_system', 'network']
    actions = ['started', 'stopped', 'restarted', 'failed', 'completed', 'timed out']
    error_messages = [
        'Disk space low',
        'Connection refused',
        'Memory allocation failed',
        'Permission denied',
        'Timeout exceeded',
        'Invalid configuration',
        'Service not responding'
    ]

    # Determine when the log should start (recent time)
    end_time = datetime.now()
    end_time_ts = end_time.timestamp()
    start_time = end_time - timedelta(hours=24)
    start_time_ts = start_time.timestamp()

    log_content = []

    for i in range(lines):
        # Generate a timestamp between start and end time
        timestamp = datetime.fromtimestamp(random.uniform(start_time_ts, end_time_ts)).strftime('%Y-%m-%d %H:%M:%S')

        # Randomly select log level with higher probability for INFO and DEBUG
        level = random.choices(
            log_levels,
            weights=[0.6, 0.2, 0.15, 0.05],  # More INFO and DEBUG messages
            k=1
        )[0]

        # Generate service and action
        service = random.choice(services)
        action = random.choice(actions)

        # Create base message
        message = f"{service}: {action}"

        # Add error details if this is an ERROR message
        if level == 'ERROR':
            error_msg = random.choice(error_messages)
            message += f" - {error_msg}"

        # Add some random details to make messages more realistic
        if random.random() < 0.3:  # 30% chance to add details
            if level in ['INFO', 'DEBUG']:
                message += f" - Process ID: {random.randint(1000, 9999)}"
            elif level == 'WARNING':
                message += f" - Retrying in {random.randint(1, 5)} seconds"

        # Create log entry
        log_entry = f"{timestamp} [{level}] {message}\n"
        log_content.append(log_entry)

    # Join all entries into a single string
    return ''.join(log_content)

# Generate log data and store in a variable
log_data = generate_system_log(lines=200)

with open ("log_entries.txt", "w") as log_file:
    log_file.write(log_data)
    log_file.close()

<p>1. Read the log_entries.txt file into your notebook, using with <code>open()</code> and print the first 10 lines to gain an understanding of the stucture of the file.

In [2]:
# Question 1 - Your code goes in this cell. Feel free to add extra cells if you need to.
with open("log_entries.txt", "r") as file:
    lines = file.readlines()

for line in lines[:10]:
    print(line.strip())


2026-08-19 18:54:41 [WARNING] database: started - Retrying in 2 seconds
2026-08-20 07:52:18 [INFO] web_server: completed
2026-08-19 15:27:22 [INFO] web_server: timed out - Process ID: 7832
2026-08-19 20:10:49 [INFO] network: stopped
2026-08-19 18:43:27 [DEBUG] network: restarted - Process ID: 6632
2026-08-19 15:02:09 [INFO] file_system: failed
2026-08-20 07:02:19 [DEBUG] database: timed out - Process ID: 6620
2026-08-19 14:53:53 [INFO] network: started
2026-08-19 22:24:33 [WARNING] file_system: completed
2026-08-20 05:05:48 [INFO] database: stopped - Process ID: 1924


<p>2. You should see that the third column on the data shows a status message. We need to extract only the lines that show as "ERROR". Using with <code>open()</code> again, write these lines to a new file.

HINT - You might find a list comprehension the most efficient way to achieve the first part of this task.

In [4]:
# Question 2 - Your code goes in this cell. Feel free to add extra cells if you need to.

with open("log_entries.txt", "r") as file:
    lines = file.readlines()

errors = [line for line in lines if "ERROR" in line]

with open("error_entries.txt", "w") as file:
    file.writelines(errors)

<h2><font color="red">Data Ingestion Challenges II</font></h2>

<p>1. Use the example URL from the slides but change the lat/lon data to your hometown or somewhere you're headed to on holiday. Anywhere really. Pull the data and explore the response<p>

```url = ("https://api.open-meteo.com/v1/forecast?latitude=51.51&longitude=-0.13&current=temperature_2m,wind_speed_10m")```<p>

Because we need to import requests for this challenge, make sure you add in to your venv. ```uv add requests``` in the terminal. You'll then need to import it in your notebook.

In [36]:
# You can copy the example from the slides and paste it here to help you get started.
%pip install requests
import requests

url = ("https://api.open-meteo.com/v1/forecast?latitude=51.2405&longitude=-0.1704&current=temperature_2m,is_day,weather_code&timezone=Europe%2FLondon")

response = requests.get(url)
data = response.json()

current = data["current"]
print("Temperature:", current["temperature_2m"], "C")
print("Wind Speed:", current["wind_speed_10m"], "km/h")


Note: you may need to restart the kernel to use updated packages.
Temperature: 14.8 C


KeyError: 'wind_speed_10m'

In [ ]:
# Question 1 - Your code goes in this cell. Feel free to add extra cells if you need to.



<h2><font color="red">Datetime Challenges</font></h2>

<p>1. Quick datetime challenge - convert the provided date string into a datetime object.<p>

HINT - datetime is part of standard python - but you will have to import it as per the slides.

In [19]:
# Question 1 - Your code goes in this cell.

from datetime import datetime, timedelta

date = "2023-12-01T12:00:00"

conv_date = datetime.strptime(date, "%Y-%m-%dT%H:%M:%S")
print(conv_date)





2023-12-01 12:00:00


<h2><font color="red">Pandas Challenges</font></h2>

<font color="red"><b>Before going any further, run ```uv add pandas``` and ```uv add faker``` in the terminal.</b></font></p>

<font color="blue"><b>Run the cell below. It's going to create a pandas dataframe named ```df``` that 'looks' a little like the dictionary you worked on.</b></font></p>


In [9]:
%pip install pandas
from IPython.display import clear_output
from faker import Faker
import pandas as pd
import random

fake = Faker(["ru_RU","zh_CN","en_US","en_GB"])
details_dict = {}
for i in range(10000):
    name = fake.name()
    email_name = random.choice(['','_','.']).join(name.split())
    domain = fake.ascii_email().split('@')[1]
    email = '@'.join([email_name, domain])
    details_dict[email] = (name, fake.date_of_birth(), fake.passport_number(), fake.ipv4(), fake.user_agent(), fake.latlng())

clear_output()

df = pd.DataFrame.from_dict(details_dict, orient='index', columns=['Name', 'Date of Birth', 'Passport Number', 'IP Address', 'User Agent', 'Location']).reset_index().rename(columns={'index':'Email'})

<p>1. Display only the top 10 rows of df <p>

In [10]:
# Question 1. Your code goes in this cell

df.head(10)


,Email,Name,Date of Birth,Passport Number,IP Address,User Agent,Location
0,杨兰英@rao.info,杨兰英,2015-09-06,90 16 303395,26.178.147.37,Opera/9.79.(X11; Linux i686; mg-MG) Presto/2.9...,"(87.538336, -105.488567)"
1,г-нМатвеевФерапонтТрофимович@mail.ru,г-н Матвеев Ферапонт Трофимович,2007-12-26,316659061,106.224.8.165,Mozilla/5.0 (Macintosh; PPC Mac OS X 10_9_6) A...,"(-7.061151, -171.655902)"
2,DrBeverleyMiller@pierce-black.net,Dr Beverley Miller,1921-12-06,87 75 467037,124.66.194.30,Mozilla/5.0 (Windows; U; Windows CE) AppleWebK...,"(-55.585267, 135.891879)"
3,Hilary_Davies@yahoo.com,Hilary Davies,1914-02-22,T88876074,151.77.184.195,Mozilla/5.0 (compatible; MSIE 7.0; Windows NT ...,"(-28.9940855, 7.744674)"
4,刘荣@yahoo.co.uk,刘荣,2004-04-29,K05657619,223.165.61.145,Mozilla/5.0 (X11; Linux i686) AppleWebKit/532....,"(51.256999, 83.158073)"
5,王洋@gangzheng.cn,王洋,1946-10-01,T97518327,103.22.177.71,Mozilla/5.0 (Android 5.0.2; Mobile; rv:24.0) G...,"(-86.923259, 51.553449)"
6,Карл_Дорофеевич_Тихонов@rambler.ru,Карл Дорофеевич Тихонов,2009-12-02,96 40 796857,82.255.151.105,Mozilla/5.0 (iPod; U; CPU iPhone OS 4_2 like M...,"(-1.712315, 165.754692)"
7,ПанфилФилатовичКонстантинов@gmail.com,Панфил Филатович Константинов,1959-01-23,T26290653,178.253.177.22,Mozilla/5.0 (compatible; MSIE 7.0; Windows NT ...,"(2.663328, -77.031606)"
8,马娟@rogers-benson.com,马娟,1913-03-05,179672038,4.206.188.238,Mozilla/5.0 (Windows NT 6.1) AppleWebKit/533.1...,"(71.923133, -119.717339)"
9,赵淑兰@yandex.ru,赵淑兰,1972-06-26,I29598933,137.12.184.148,Mozilla/5.0 (Windows 98; Win 9x 4.90) AppleWeb...,"(25.588494, 133.322786)"


<p>2. Display only the last 10 rows of df <p>

In [11]:
# Question 2. Your code goes in this cell.

df.tail(10)


,Email,Name,Date of Birth,Passport Number,IP Address,User Agent,Location
9959,ЕвгенияВадимовнаАнисимова@hotmail.com,Евгения Вадимовна Анисимова,2017-12-06,B31068486,126.45.22.89,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_7_3;...,"(-60.0875785, -144.668737)"
9960,Damien_Cartwright@hotmail.com,Damien Cartwright,1964-11-08,641338940,214.228.235.152,Mozilla/5.0 (Macintosh; U; PPC Mac OS X 10_10_...,"(-61.7788315, -54.364355)"
9961,徐磊@weston-mitchell.com,徐磊,1989-07-31,J92274121,16.210.52.130,Opera/9.92.(X11; Linux x86_64; mni-IN) Presto/...,"(32.062385, 0.154557)"
9962,Jenna.Knight@hotmail.com,Jenna Knight,1936-09-05,234794143,194.171.200.50,Mozilla/5.0 (compatible; MSIE 8.0; Windows NT ...,"(-86.7165745, -148.905518)"
9963,Irene_King@ellis.org,Irene King,1988-07-15,T19210411,94.166.115.161,Mozilla/5.0 (Linux; Android 2.3.3) AppleWebKit...,"(-32.080131, 127.977070)"
9964,АрхиповТрифонАлександрович@yudin.info,Архипов Трифон Александрович,1967-10-08,770288084,23.116.172.4,Mozilla/5.0 (Windows; U; Windows NT 5.01) Appl...,"(-69.6830335, -147.953668)"
9965,Brittney_Ayers@yahoo.co.uk,Brittney Ayers,1921-07-23,924290654,183.53.167.215,Mozilla/5.0 (iPhone; CPU iPhone OS 7_1_2 like ...,"(-89.797809, -84.440914)"
9966,Корнил.Васильевич.Воронцов@hotmail.com,Корнил Васильевич Воронцов,2014-05-02,39 35 734741,202.102.217.171,Mozilla/5.0 (Windows 95; kl-GL; rv:1.9.2.20) G...,"(71.6383365, -103.046744)"
9967,VincentBrennan@ip.org,Vincent Brennan,2004-09-24,A91739982,184.226.231.67,Mozilla/5.0 (Macintosh; U; PPC Mac OS X 10_5_0...,"(28.7878165, 132.687444)"
9968,张莉@oao.ru,张莉,2018-04-04,4245 065483,89.180.26.8,Opera/9.10.(Windows NT 5.1; hsb-DE) Presto/2.9...,"(-27.439018, -170.250620)"


<p>3. Display only the 'name' column of df <p>

In [17]:
# Question 3. Your code goes in this cell.

df["Name"]

0                                   杨兰英
1       г-н Матвеев Ферапонт Трофимович
2                    Dr Beverley Miller
3                         Hilary Davies
4                                    刘荣
                     ...               
9964       Архипов Трифон Александрович
9965                     Brittney Ayers
9966         Корнил Васильевич Воронцов
9967                    Vincent Brennan
9968                                 张莉
Name: Name, Length: 9969, dtype: str


<p>4. Using <code>.apply()</code>, create and apply a lambda to calculate the age of each individual and create a new column to display this data.

In [30]:
# Question 4. HINT - as in the slides, consider using timedelta to calculate the difference between now and the DOB of each individual.
from datetime import datetime, timedelta

df["Age"] = df["Date of Birth"].apply(lambda x: datetime.now().date() - x).days//365.25)

SyntaxError: unmatched ')' (3124511325.py, line 4)

<p>5. Using the columns you've just created, output the names and ages of only those aged 40 years or more.

In [32]:
# Question 5 - Your code goes in this cell.
df[df["Age"]>40.0][["Name", "Age"]]


TypeError: Invalid comparison between dtype=timedelta64[us] and float

<p>6. Break out the lat/lon column into to two separate columns using .apply() with a function that you'll need to define.

In [ ]:
# Question 6 - Your code goes in this cell.

def lat_lon(location):
    lat = location[0]
    lon = location[1]
    return lat, lon

df[["lat", "lon"]] = df["Location"]

<p><br><br><font color='yellow'> Well done! You've reached the end of Workbook 2.</font>

For the remainder of the time we have together, we'll be working on a capstone project of your choosing.

Some suggesions:<p>
<ul>
<li>A hangman game - This is the old challenge j-hub challenge to earn the Python skill. We've learned everything you need to make this well.</li>
<li>A weather reporting script - We've looked at using an API to get weather data. You can build on this and make it as complicated as you like.<P> 
<p>Talk to me about plotting graphs, maps or creating dashboards. This one could get crazy.<p>
<p>You could create look back analysis to compare historic data with what we're seeing today. Dig into the data from the API and scour the docs. There's a lot there.</li>
</ul>


## Household Weather Helper

## Purpose:
Use live weather data to give simple household recommendation about whether conditions are suitable for drying laundry outside.

## Input:
- Predefined location
- Weather data from Open-Meteo

## Weather information needed:
- Temperature
- Chance of rain

## Process:
1. Request weather data from Open-Meteo.
2. Extract temperature.
3. Extract chance of rain.
4. Pass those values into a function.
5. Use if/elif/else to decide recommendation:
    - Rain probability >= 60% --> recommend drying indoors
    - Rain probability < 60% and temperature < 5°C --> recommend outdoor drying is possible but not ideal
    - Rain probablilty < 60% and temperature > 5°C --> recommend good conditions for outdoor drying

## Output:
- Location
- Temperature
- Chance of rain
- Household recommendation

In [ ]:
import requests

location = "Lincoln"
latitude = 53.23
longitude = -0.54

url = (
    f"https://api.open-meteo.com/v1/forecast"
    f"?latitude={latitude}&longitude={longitude}"
    f"&current=temperature_2m,precipitation_probability"
)
    
response = requests.get(url)

data = response.json()

temperature = data["current"]["temperature_2m"]
rain_probability = data["current"]["precipitation_probability"]

def weather_advice(temperature, rain_probability):
    if rain_probability >= 60:
        return "Dry indoors"

print(weather_advice(15, 75))


 






Dry indoors


In [ ]:
# import requests

# url = (
#     "https://geocoding-api.open-meteo.com/v1/search/"
# )

# params = {
#     "name": "Berlin",
# }

# resp = requests.get(url, params=params).json()

# print(resp)